<a href="https://colab.research.google.com/github/noufanelachola/code_to_language/blob/main/CodeToEnglish_Translator_Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# AI-Powered Code-to-English Translator for Healthcare Software
# Main Notebook - CodeBERT + Gradio
# Name: Noufan Elachola | Roll No: 69 | Class: s6 CS2

In [ ]:
!pip install -q transformers torch gradio

In [ ]:
from transformers import AutoTokenizer, AutoModel

In [ ]:
import torch

In [ ]:
MODEL_NAME = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

In [ ]:
print("Loaded model:", MODEL_NAME)

In [ ]:
def get_code_embedding(code_snippet: str) -> torch.Tensor:
    inputs = tokenizer(code_snippet, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)  # last_hidden_state: (batch, seq_len, hidden)
    embedding = outputs.last_hidden_state.mean(dim=1)  # simple mean pooling
    return embedding[0]

In [ ]:
sample_code = """
def calculate_bmi(weight_kg, height_m):
    if height_m <= 0:
        raise ValueError("Height must be greater than zero.")

    bmi = weight_kg / (height_m ** 2)
    return round(bmi, 2)
"""

In [ ]:
get_code_embedding(sample_code)

In [11]:
import ast
from textwrap import dedent

In [12]:
def explain_code(code_str: str) -> str:
    if not code_str.strip():
        return "No code provided. Please paste a Python function or logic."

    code_str = dedent(code_str)

    try:
        tree = ast.parse(code_str)
    except SyntaxError:
        return "The input is not valid Python code. Please check for syntax errors."

    lines = []

    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef):
            func_name = node.name
            params = [arg.arg for arg in node.args.args]
            if params:
                lines.append(
                    f"This code defines a function named '{func_name}' that takes the parameters: {', '.join(params)}."
                )
            else:
                lines.append(
                    f"This code defines a function named '{func_name}' that does not take any parameters."
                )

    for node in ast.walk(tree):
        if isinstance(node, ast.If):
            try:
                cond = ast.unparse(node.test)
            except Exception:
                cond = "<condition>"
            lines.append(f"It checks the condition: {cond}.")

    for node in ast.walk(tree):
        if isinstance(node, ast.Return):
            try:
                value = ast.unparse(node.value)
            except Exception:
                value = "<expression>"
            lines.append(f"It returns the value: {value}.")

    if not lines:
        lines.append(
            "This code contains basic Python statements such as assignments, expressions, or function calls."
        )

    return " ".join(lines)


In [13]:
explain_code(sample_code)

"This code defines a function named 'calculate_bmi' that takes the parameters: weight_kg, height_m. It checks the condition: height_m <= 0. It returns the value: round(bmi, 2)."